# Ablation Study — GLAAM-4X

This notebook systematically removes or replaces each component of GLAAM-4X to measure its individual contribution. The results directly answer the reviewer question: **"Does each component actually help?"**

## Ablation Configurations

| # | Name | What Changes | What We're Testing |
|---|------|-------------|-------------------|
| 1 | **Full GLAAM-4X** | Nothing (baseline) | Reference — best model |
| 2 | **No MultiScale** | Replace MultiScaleGLAAM (DR head) with standard GLAAMBlock | Does multi-scale attention help DR? |
| 3 | **No Disease Gating** | Remove DiseaseGatingNetwork, use equal weights | Does learned routing help? |
| 4 | **No ASL** | Replace Asymmetric Loss with BCEWithLogitsLoss | Does ASL help vs standard BCE? |
| 5 | **No Strong Aug** | Disable strong augmentation for minority class (70%→0%) | Does differential augmentation help? |
| 6 | **No Attention (Myopia for all)** | Replace all attention heads with Identity | Does attention help at all? |
| 7 | **Shared Attention** | Use one GLAAMBlock for all diseases (no specialists) | Do disease-specific heads beat shared? |
| 8 | **No Warmup** | Remove 10-epoch warmup, start cosine immediately | Does warmup help? |

## How It Works

Each ablation variant:
1. Modifies the model or training config
2. Trains for a **reduced 20 epochs** (enough to see trends, saves Colab time)
3. Evaluates on the **same validation + test split** as the full model
4. Records macro F1, per-disease AUC/F1, and parameter count

At the end, all variants are compared in a summary table and bar chart.

## Prerequisites

- Run this on **Google Colab with GPU**
- Your Drive must have the same `DRIVE_BASE` structure as the training notebook
- The data CSVs (`train_v4.csv`, `val_tune_v4.csv`, `test_v4.csv`) must already exist in `/tmp/data/` — **run Sections 1–3 of the training notebook first** to download and parse the data, then come here

In [ ]:
import subprocess
import sys
import os
from pathlib import Path

# ═══════════════════════════════════════════════════════════════
# Section 1: Setup — assumes training notebook Sections 1-3 already run
# (deps installed, Drive mounted, data downloaded and parsed)
# ═══════════════════════════════════════════════════════════════

# Verify data exists from training notebook
DATA_DIR = Path("/tmp/data")
for csv_name in ["train_v4.csv", "val_tune_v4.csv", "test_v4.csv"]:
    csv_path = DATA_DIR / csv_name
    if not csv_path.exists():
        raise FileNotFoundError(
            f"{csv_path} not found! Run Sections 1-3 of the training notebook "
            f"(modal_train_glaam4x_v4_kaggle) first to download and parse data."
        )

# Verify Drive is mounted (for saving results)
DRIVE_BASE = "/content/drive/MyDrive/Backup/dataset/cataract_detection"
if not os.path.exists(DRIVE_BASE):
    from google.colab import drive
    drive.mount('/content/drive')

sys.path.insert(0, DRIVE_BASE)

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torchvision import models, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from tqdm import tqdm
import json
import time

DISEASE_NAMES = ['Cataract', 'DR', 'Glaucoma', 'Myopia']
IMG_SIZE = 384
BATCH_SIZE = 32
ABLATION_EPOCHS = 20  # reduced from 60 for speed
WARMUP_EPOCHS = 5     # reduced from 10 proportionally

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Data CSVs found in {DATA_DIR}")
print(f"Ablation epochs: {ABLATION_EPOCHS} (reduced for speed)")
print(f"Drive base: {DRIVE_BASE}")

# Load data
train_df = pd.read_csv(DATA_DIR / "train_v4.csv")
val_df = pd.read_csv(DATA_DIR / "val_tune_v4.csv")
test_df = pd.read_csv(DATA_DIR / "test_v4.csv")
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# Section 2: Shared Components

These functions are reused across all ablation variants. Only the model architecture and training config change per variant.

In [ ]:
import cv2
from PIL import Image
from torchvision.transforms import v2

# ── Dataset (same as training notebook, with configurable augmentation) ──────
class GaussianNoise(torch.nn.Module):
    def __init__(self, std_range=(0.04, 0.2), p=0.3):
        super().__init__()
        self.std_range = std_range
        self.p = p
    def forward(self, img):
        if torch.rand(1).item() > self.p:
            return img
        std = torch.empty(1).uniform_(*self.std_range).item()
        noise = torch.randn_like(img) * std
        return torch.clamp(img + noise, 0.0, 1.0)


def get_transforms(img_size, is_train, strong_aug=True):
    base_tail = [
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
    if not is_train:
        return v2.Compose([v2.Resize((img_size, img_size))] + base_tail)

    train_transforms = [
        v2.Resize((img_size, img_size)),
        v2.RandomHorizontalFlip(p=0.5),
        v2.RandomVerticalFlip(p=0.3),
        v2.RandomApply([v2.RandomChoice([
            v2.RandomRotation((90, 90)),
            v2.RandomRotation((180, 180)),
            v2.RandomRotation((270, 270)),
        ])], p=0.3),
        v2.RandomAffine(degrees=15, translate=(0.05, 0.05), scale=(0.9, 1.1)),
        v2.RandomApply([v2.ColorJitter(brightness=0.2, contrast=0.2)], p=0.5),
        v2.RandomApply([v2.ColorJitter(hue=0.03, saturation=0.2)], p=0.3),
        v2.RandomApply([v2.GaussianBlur(kernel_size=3)], p=0.2),
        v2.RandomApply([v2.ElasticTransform(alpha=50.0, sigma=5.0)], p=0.1),
    ] + base_tail + [
        GaussianNoise(std_range=(0.04, 0.2), p=0.3),
        v2.RandomErasing(p=0.2, scale=(0.01, 0.05), ratio=(0.5, 2.0)),
    ]

    if strong_aug:
        strong_tail = [
            v2.ToImage(), v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
        strong_transforms = v2.Compose([
            v2.Resize((img_size, img_size)),
            v2.RandomHorizontalFlip(p=0.5),
            v2.RandomVerticalFlip(p=0.3),
            v2.RandomApply([v2.RandomChoice([
                v2.RandomRotation((90, 90)),
                v2.RandomRotation((180, 180)),
                v2.RandomRotation((270, 270)),
            ])], p=0.3),
            v2.RandomAffine(degrees=30, translate=(0.1, 0.1), scale=(0.8, 1.2)),
            v2.RandomApply([v2.ColorJitter(brightness=0.3, contrast=0.3)], p=0.5),
            v2.RandomApply([v2.ColorJitter(hue=0.05, saturation=0.3)], p=0.4),
            v2.RandomApply([v2.GaussianBlur(kernel_size=5)], p=0.3),
            v2.RandomApply([v2.ElasticTransform(alpha=100.0, sigma=5.0)], p=0.2),
        ] + strong_tail + [
            GaussianNoise(std_range=(0.08, 0.3), p=0.4),
            v2.RandomErasing(p=0.3, scale=(0.02, 0.08), ratio=(0.5, 2.0)),
        ])
    else:
        strong_transforms = None

    return train_transforms, strong_transforms


class UnifiedDataset(torch.utils.data.Dataset):
    def __init__(self, df, img_dir, disease_cols, img_size, is_train=False,
                 strong_aug=True, minority_aug_prob=0.7):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.disease_cols = disease_cols
        self.is_train = is_train
        self.minority_aug_prob = minority_aug_prob if strong_aug else 0.0
        self.transform, self.strong_aug = get_transforms(img_size, is_train, strong_aug)

    def __len__(self):
        return len(self.df)

    def _load_image(self, path):
        if not os.path.isabs(path):
            for p in [os.path.join(self.img_dir, path), os.path.join(self.img_dir, "raw", path)]:
                if os.path.exists(p):
                    path = p
                    break
        img = cv2.imread(path)
        if img is None:
            raise FileNotFoundError(f"Image not found: {path}")
        return Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self._load_image(row['image_path'])
        labels = row[self.disease_cols].values.astype(np.float32)
        if self.is_train and labels.sum() > 0 and np.random.rand() < self.minority_aug_prob:
            img = self.strong_aug(img)
        else:
            img = self.transform(img)
        return img, torch.tensor(labels)


# ── Metrics ──────────────────────────────────────────────────────────────────
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

def compute_metrics(logits, labels, thresholds=None):
    probs = 1 / (1 + np.exp(-logits))
    if thresholds is None:
        thresholds = {d: 0.5 for d in DISEASE_NAMES}
    metrics = {}
    for i, disease in enumerate(DISEASE_NAMES):
        y_true = labels[:, i]
        y_prob = probs[:, i]
        thr = thresholds.get(disease, 0.5)
        y_pred = (y_prob >= thr).astype(int)
        metrics[disease] = {
            'auc': roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else 0.5,
            'f1': f1_score(y_true, y_pred, zero_division=0),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
        }
    metrics['macro_f1'] = np.mean([m['f1'] for m in metrics.values()])
    return metrics, probs


def find_optimal_thresholds(logits, labels):
    probs = 1 / (1 + np.exp(-logits))
    thresholds = {}
    for i, disease in enumerate(DISEASE_NAMES):
        y_true = labels[:, i]
        y_prob = probs[:, i]
        best_f1, best_thr = 0, 0.5
        for thr in np.arange(0.05, 0.95, 0.01):
            y_pred = (y_prob >= thr).astype(int)
            f1 = f1_score(y_true, y_pred, zero_division=0)
            if f1 > best_f1:
                best_f1, best_thr = f1, thr
        thresholds[disease] = float(best_thr)
    return thresholds


# ── Training & evaluation functions ─────────────────────────────────────────
scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

def train_epoch(model, loader, criterion, optimizer, scheduler):
    model.train()
    total_loss = 0.0
    all_logits, all_labels = [], []
    for images, labels in tqdm(loader, desc="Training", leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        if scaler is not None:
            with torch.amp.autocast('cuda'):
                logits = model(images)
                loss = criterion(logits, labels)
            if not torch.isfinite(loss):
                scaler.update()
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(images)
            loss = criterion(logits, labels)
            if not torch.isfinite(loss):
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
            optimizer.step()
        total_loss += loss.item()
        all_logits.append(logits.detach().cpu())
        all_labels.append(labels.cpu())
    scheduler.step()
    return total_loss / len(loader), torch.cat(all_logits).numpy(), torch.cat(all_labels).numpy()


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    for images, labels in tqdm(loader, desc="Evaluating", leave=False):
        images = images.to(device)
        logits = model(images)
        all_logits.append(logits.cpu())
        all_labels.append(labels)
    return torch.cat(all_logits).numpy(), torch.cat(all_labels).numpy()


# ── Sampler ──────────────────────────────────────────────────────────────────
def get_sample_weights(df, disease_cols):
    pos_counts = df[disease_cols].sum().values
    neg_counts = len(df) - pos_counts
    pos_weights = np.sqrt(1.0 / (pos_counts + 1e-6))
    neg_weights = np.sqrt(1.0 / (neg_counts + 1e-6))
    pos_weights = pos_weights / pos_weights.sum()
    neg_weights = neg_weights / neg_weights.sum()
    weights = np.zeros(len(df))
    for i, row in df.iterrows():
        w = 0.0
        for j, col in enumerate(disease_cols):
            w += pos_weights[j] if row[col] == 1 else neg_weights[j]
        weights[i] = w / len(disease_cols)
    return weights


# ── Full ablation runner ─────────────────────────────────────────────────────
import multiprocessing
NUM_WORKERS = min(2, multiprocessing.cpu_count() - 1)

def run_ablation(name, model, criterion_fn, strong_aug=True, use_warmup=True,
                 drive_base=DRIVE_BASE, save_results=True):
    """Train one ablation variant and return results dict."""
    print(f"\n{'='*60}")
    print(f"  ABLATION: {name}")
    print(f"{'='*60}")

    # Count parameters
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Parameters: {n_params:,}")

    model = model.to(device)

    # Datasets
    train_ds = UnifiedDataset(train_df, str(DATA_DIR), DISEASE_NAMES, IMG_SIZE,
                              is_train=True, strong_aug=strong_aug)
    val_ds = UnifiedDataset(val_df, str(DATA_DIR), DISEASE_NAMES, IMG_SIZE, is_train=False)
    test_ds = UnifiedDataset(test_df, str(DATA_DIR), DISEASE_NAMES, IMG_SIZE, is_train=False)

    sample_weights = get_sample_weights(train_df, DISEASE_NAMES)
    sampler = WeightedRandomSampler(
        weights=torch.tensor(sample_weights, dtype=torch.double),
        num_samples=len(train_df) * 2, replacement=True)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=NUM_WORKERS, pin_memory=True,
                              prefetch_factor=2, persistent_workers=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)

    # Optimizer & scheduler
    optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=5e-4)
    warmup = WARMUP_EPOCHS if use_warmup else 0
    def lr_lambda(epoch):
        if epoch < warmup:
            return float(epoch + 1) / float(max(1, warmup))
        progress = (epoch - warmup) / max(1, ABLATION_EPOCHS - warmup)
        return 0.5 * (1.0 + np.cos(np.pi * progress))
    scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)

    criterion = criterion_fn()

    # Training loop
    best_val_f1 = 0.0
    best_epoch = 0
    history = {'epoch': [], 'val_macro_f1': [], 'train_loss': []}

    for epoch in range(1, ABLATION_EPOCHS + 1):
        train_loss, train_logits, train_labels = train_epoch(
            model, train_loader, criterion, optimizer, scheduler)
        val_logits, val_labels = evaluate(model, val_loader)
        val_metrics, _ = compute_metrics(val_logits, val_labels)
        opt_thr = find_optimal_thresholds(val_logits, val_labels)
        val_metrics_opt, _ = compute_metrics(val_logits, val_labels, opt_thr)

        history['epoch'].append(epoch)
        history['val_macro_f1'].append(val_metrics_opt['macro_f1'])
        history['train_loss'].append(train_loss)

        if val_metrics_opt['macro_f1'] > best_val_f1:
            best_val_f1 = val_metrics_opt['macro_f1']
            best_epoch = epoch
            best_thresholds = opt_thr

        if epoch % 5 == 0 or epoch == ABLATION_EPOCHS:
            print(f"  Epoch {epoch}/{ABLATION_EPOCHS} | Loss: {train_loss:.4f} | "
                  f"Val F1: {val_metrics_opt['macro_f1']:.4f} (best: {best_val_f1:.4f})")

    # Final test evaluation with best thresholds
    test_logits, test_labels = evaluate(model, test_loader)
    test_metrics, _ = compute_metrics(test_logits, test_labels, best_thresholds)

    print(f"\n  ✅ {name} — Best Val F1: {best_val_f1:.4f} (epoch {best_epoch})")
    print(f"  Test Macro F1: {test_metrics['macro_f1']:.4f}")
    for d in DISEASE_NAMES:
        m = test_metrics[d]
        print(f"    {d:12s} AUC={m['auc']:.4f} F1={m['f1']:.4f}")

    result = {
        'name': name,
        'n_params': n_params,
        'best_val_f1': best_val_f1,
        'best_epoch': best_epoch,
        'test_macro_f1': test_metrics['macro_f1'],
        'test_metrics': test_metrics,
        'best_thresholds': best_thresholds,
        'history': history,
    }

    if save_results:
        save_dir = Path(drive_base) / "ablation_results"
        save_dir.mkdir(parents=True, exist_ok=True)
        with open(save_dir / f"{name.replace(' ', '_')}.json", 'w') as f:
            json.dump(result, f, indent=2, default=str)
        print(f"  Saved to {save_dir / f'{name.replace(' ', '_')}.json'}")

    # Free GPU memory
    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    return result

print("✅ Shared components defined.")

# Section 3: Ablation Model Variants

Each variant modifies the GLAAM-4X architecture or training config to test one hypothesis. The `REORDER_IDX` wrapper is reused to keep the output order consistent.